# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path
import duckdb
import polars as pl

WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH}")

df = pl.scan_csv(DATA_PATH, infer_schema_length=0)

print("Lokasi proyek:", PROJECT_ROOT)
print("Lokasi dataset:", DATA_PATH)
print("Daftar kolom:", df.collect_schema().names())

df.limit(5).collect()


Lokasi proyek: /home/jovyan/work
Lokasi dataset: /home/jovyan/work/data/raw/data.csv
Daftar kolom: ['Label', 'Denomination', 'Location', 'Date', 'Title', 'Sentence_Unit', 'Link']


Label,Denomination,Location,Date,Title,Sentence_Unit,Link
str,str,str,str,str,str,str
"""Catholic""","""Katolisitas""","""National""",null,null,"""Banyak orang yang bertanya-tan…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Sebab ketika berselingkuh, ses…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Kalian pengen masuk Surga ga?""","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Tinggallah bersama kami, ya Tu…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Perjalanan dua orang murid ke …","""https://katolisitas.org"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [5]:
connection = duckdb.connect()

profil = connection.execute(
    "SUMMARIZE SELECT * FROM read_csv(?, all_varchar=true)",
    [str(DATA_PATH)]
).pl()

profil


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""Label""","""VARCHAR""","""Catholic""","""Protestant""",3,null,null,null,null,null,3013172,0.00
"""Denomination""","""VARCHAR""","""Alkitab SABDA""","""Yufid""",68,null,null,null,null,null,3013172,0.00
"""Location""","""VARCHAR""","""International""","""Sumatra""",10,null,null,null,null,null,3013172,0.00
"""Date""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.00
"""Title""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.00
"""Sentence_Unit""","""VARCHAR""","""( (21): 87) Inilah tasbih Nabi…","""𝐅𝐫𝐚𝐧𝐬 𝐒𝐢𝐧𝐮𝐫𝐚𝐭Mengucapkan terim…",1963349,null,null,null,null,null,3013172,0.00
"""Link""","""VARCHAR""","""http://Www.muslimah.or.id""","""https://zakat.ldii.or.id/zakat…",128940,null,null,null,null,null,3013172,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [6]:
nama_kolom = df.collect_schema().names()

stat_null = df.select(
    pl.len().alias("total_baris"),
    *[pl.col(k).null_count().alias(k) for k in nama_kolom]
).collect()

total_baris = stat_null["total_baris"][0]

hasil_null = pl.DataFrame({
    "kolom": nama_kolom,
    "jumlah_null": [stat_null[k][0] for k in nama_kolom]
}).with_columns(
    (pl.col("jumlah_null") / total_baris * 100)
    .round(2)
    .alias("persentase_null")
)

print("Total baris:", total_baris)
hasil_null


Total baris: 3013172


kolom,jumlah_null,persentase_null
str,i64,f64
"""Label""",0,0.0
"""Denomination""",0,0.0
"""Location""",0,0.0
"""Date""",3013172,100.0
"""Title""",3013172,100.0
"""Sentence_Unit""",0,0.0
"""Link""",0,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [7]:
kolom_kosong = hasil_null.filter(
    pl.col("persentase_null") == 100
)["kolom"].to_list()

print("Kolom yang seluruhnya NULL:", kolom_kosong)

df_clean = df.drop(kolom_kosong)

ringkasan_clean = df_clean.select(
    pl.len().alias("jumlah_baris"),
    pl.all().null_count()
).collect()

print("Kolom yang tersisa:", df_clean.collect_schema().names())
ringkasan_clean


Kolom yang seluruhnya NULL: ['Date', 'Title']
Kolom yang tersisa: ['Label', 'Denomination', 'Location', 'Sentence_Unit', 'Link']


jumlah_baris,Label,Denomination,Location,Sentence_Unit,Link
u32,u32,u32,u32,u32,u32
3013172,0,0,0,0,0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [8]:
df_panjang = df_clean.with_columns(
    pl.col("Sentence_Unit")
    .str.len_chars()
    .alias("panjang_teks")
)

stat_panjang = df_panjang.select(
    pl.col("panjang_teks").min().alias("minimum"),
    pl.col("panjang_teks").quantile(0.25).alias("Q1"),
    pl.col("panjang_teks").median().alias("median"),
    pl.col("panjang_teks").quantile(0.75).alias("Q3"),
    pl.col("panjang_teks").max().alias("maksimum")
).collect()

q1 = stat_panjang["Q1"][0]
q3 = stat_panjang["Q3"][0]

iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

hasil_outlier = df_panjang.select(
    (pl.col("panjang_teks") < batas_bawah)
    .sum().alias("outlier_pendek"),

    (pl.col("panjang_teks") > batas_atas)
    .sum().alias("outlier_panjang"),

    (pl.col("Sentence_Unit").str.strip_chars() == "")
    .sum().alias("teks_kosong")
).collect()

print("Statistik panjang teks:")
print(stat_panjang)

print("\nBatas bawah IQR:", batas_bawah)
print("Batas atas IQR:", batas_atas)

print("\nHasil deteksi outlier:")
hasil_outlier


Statistik panjang teks:
shape: (1, 5)
┌─────────┬──────┬────────┬───────┬──────────┐
│ minimum ┆ Q1   ┆ median ┆ Q3    ┆ maksimum │
│ ---     ┆ ---  ┆ ---    ┆ ---   ┆ ---      │
│ u32     ┆ f64  ┆ f64    ┆ f64   ┆ u32      │
╞═════════╪══════╪════════╪═══════╪══════════╡
│ 25      ┆ 75.0 ┆ 108.0  ┆ 154.0 ┆ 500      │
└─────────┴──────┴────────┴───────┴──────────┘

Batas bawah IQR: 0
Batas atas IQR: 272.5

Hasil deteksi outlier:


outlier_pendek,outlier_panjang,teks_kosong
u32,u32,u32
0,104083,0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [9]:
kolom_kunci = [
    "Label",
    "Denomination",
    "Location",
    "Sentence_Unit",
    "Link"
]

jumlah_awal = total_baris

jumlah_unik = (
    df_clean
    .unique(subset=kolom_kunci)
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

jumlah_duplikat = jumlah_awal - jumlah_unik

print("Jumlah baris awal:", jumlah_awal)
print("Jumlah baris unik:", jumlah_unik)
print("Jumlah duplikat:", jumlah_duplikat)


Jumlah baris awal: 3013172
Jumlah baris unik: 3012817
Jumlah duplikat: 355


In [10]:
kolom_kunci_bisnis = [
    "Label",
    "Denomination",
    "Sentence_Unit",
    "Link"
]

jumlah_unik_bisnis = (
    df_clean
    .unique(subset=kolom_kunci_bisnis)
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

duplikat_bisnis = total_baris - jumlah_unik_bisnis

konflik_lokasi = (
    df_clean
    .group_by(kolom_kunci_bisnis)
    .agg(pl.col("Location").n_unique().alias("jumlah_lokasi"))
    .filter(pl.col("jumlah_lokasi") > 1)
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

print("Jumlah baris awal:", total_baris)
print("Jumlah baris unik berdasarkan kunci bisnis:", jumlah_unik_bisnis)
print("Jumlah duplikat berdasarkan kunci bisnis:", duplikat_bisnis)
print("Jumlah kelompok dengan konflik lokasi:", konflik_lokasi)


Jumlah baris awal: 3013172
Jumlah baris unik berdasarkan kunci bisnis: 3012817
Jumlah duplikat berdasarkan kunci bisnis: 355
Jumlah kelompok dengan konflik lokasi: 0


In [11]:
cek_duplikat = (
    df_clean
    .group_by(kolom_kunci_bisnis)
    .agg(
        pl.len().alias("jumlah_baris_grup"),
        pl.col("Location").n_unique().alias("jumlah_lokasi_unik")
    )
    .select(
        (pl.col("jumlah_baris_grup") - 1)
        .sum()
        .alias("duplikat_kunci_bisnis"),

        (
            pl.col("jumlah_baris_grup") -
            pl.col("jumlah_lokasi_unik")
        )
        .sum()
        .alias("duplikat_5_kolom"),

        (pl.col("jumlah_lokasi_unik") > 1)
        .sum()
        .alias("konflik_lokasi")
    )
    .collect(engine="streaming")
)

print("Hasil verifikasi deduplikasi:")
cek_duplikat


Hasil verifikasi deduplikasi:


duplikat_kunci_bisnis,duplikat_5_kolom,konflik_lokasi
u32,u32,u32
355,355,0


In [12]:
df_dedup = df_clean.unique(
    subset=kolom_kunci_bisnis,
    keep="first"
)

jumlah_setelah = (
    df_dedup
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

jumlah_terhapus = total_baris - jumlah_setelah

assert jumlah_setelah == jumlah_unik_bisnis, (
    "Hasil deduplikasi tidak sesuai pemeriksaan sebelumnya"
)

print("Jumlah baris sebelum:", total_baris)
print("Jumlah baris setelah:", jumlah_setelah)
print("Jumlah duplikat dihapus:", jumlah_terhapus)
print("Jumlah kolom:", len(df_dedup.collect_schema().names()))


Jumlah baris sebelum: 3013172
Jumlah baris setelah: 3012817
Jumlah duplikat dihapus: 355
Jumlah kolom: 5


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [13]:
kolom_wajib = [
    "Label",
    "Denomination",
    "Location",
    "Sentence_Unit",
    "Link"
]

validasi = df_dedup.select(
    pl.len().alias("total_baris"),

    *[
        (
            pl.col(k).fill_null("").str.strip_chars() == ""
        ).sum().alias(f"kosong_{k}")
        for k in kolom_wajib
    ],

    (
        ~pl.col("Link")
        .fill_null("")
        .str.contains(r"^https?://")
    ).sum().alias("link_format_tidak_sesuai"),

    pl.col("Label").n_unique().alias("jumlah_label_unik")
).collect(engine="streaming")

print("Hasil Validasi Data Bersih:")
validasi


Hasil Validasi Data Bersih:


total_baris,kosong_Label,kosong_Denomination,kosong_Location,kosong_Sentence_Unit,kosong_Link,link_format_tidak_sesuai,jumlah_label_unik
u32,u32,u32,u32,u32,u32,u32,u32
3012817,0,0,0,0,0,0,3


In [14]:
distribusi_label = (
    df_dedup
    .group_by("Label")
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort("jumlah_baris", descending=True)
    .collect(engine="streaming")
)

print("Distribusi kategori Label:")
print(distribusi_label)

assert distribusi_label["jumlah_baris"].sum() == jumlah_setelah

print("\nValidasi jumlah baris kategori berhasil.")


Distribusi kategori Label:
shape: (3, 2)
┌────────────┬──────────────┐
│ Label      ┆ jumlah_baris │
│ ---        ┆ ---          │
│ str        ┆ u32          │
╞════════════╪══════════════╡
│ Islam      ┆ 1455099      │
│ Catholic   ┆ 797131       │
│ Protestant ┆ 760587       │
└────────────┴──────────────┘

Validasi jumlah baris kategori berhasil.


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_dedup.sink_parquet(
    OUTPUT_PATH,
    compression="zstd"
)

print("Dataset berhasil disimpan!")
print("Lokasi file:", OUTPUT_PATH)

ukuran_mb = OUTPUT_PATH.stat().st_size / (1024 ** 2)
print(f"Ukuran file: {ukuran_mb:.2f} MB")

df_parquet = pl.scan_parquet(OUTPUT_PATH)

jumlah_parquet = df_parquet.select(
    pl.len()
).collect().item()

jumlah_kolom = len(df_parquet.collect_schema().names())

print("Jumlah baris:", jumlah_parquet)
print("Jumlah kolom:", jumlah_kolom)

assert jumlah_parquet == jumlah_setelah
assert jumlah_kolom == 5

print("Validasi file Parquet berhasil!")


Dataset berhasil disimpan!
Lokasi file: /home/jovyan/work/data/processed/dataset_bersih.parquet
Ukuran file: 173.73 MB
Jumlah baris: 3012817
Jumlah kolom: 5
Validasi file Parquet berhasil!



## Catatan Milestone 2

### 1. Dataset
Dataset yang digunakan adalah Indonesian Religious Corpus, dengan jumlah awal 3.013.172 baris dan 7 kolom.

### 2. Profiling dan Missing Values
Profiling dilakukan menggunakan DuckDB SUMMARIZE dan Polars.

Ditemukan dua kolom, yaitu Date dan Title, yang memiliki missing values sebesar 100%.

Kedua kolom dihapus karena tidak memiliki informasi yang dapat digunakan. Lima kolom lainnya dipertahankan, yaitu Label, Denomination, Location, Sentence_Unit, dan Link.

### 3. Deteksi dan Penanganan Outlier
Outlier dideteksi menggunakan metode Interquartile Range (IQR) berdasarkan panjang karakter pada kolom Sentence_Unit.

Hasil perhitungan:
- Q1: 75 karakter
- Median: 108 karakter
- Q3: 154 karakter
- Batas atas IQR: 272,5 karakter
- Kandidat outlier panjang: 104.083 baris
- Teks kosong: 0 baris

Kandidat outlier dipertahankan karena panjang teks tidak secara langsung membuktikan kesalahan data.

### 4. Deduplikasi
Deduplikasi dilakukan berdasarkan kunci bisnis:
Label, Denomination, Sentence_Unit, dan Link.

Hasil akhir:
- Baris sebelum deduplikasi: 3.013.172
- Duplikat yang dihapus: 355
- Baris setelah deduplikasi: 3.012.817

Pemeriksaan kunci bisnis tidak menemukan konflik pada kolom Location.

### 5. Validasi Kualitas Data
Validasi dilakukan untuk memeriksa nilai NULL, teks kosong, whitespace, dan awalan format URL.

Hasil validasi:
- Nilai kosong pada kolom wajib: 0
- Link dengan awalan URL tidak sesuai: 0
- Jumlah kategori Label: 3

Kategori Label terdiri dari Islam, Catholic, dan Protestant.

### 6. Penyimpanan Dataset
Dataset hasil cleaning disimpan menggunakan Polars dalam format Parquet dengan kompresi ZSTD.

Lokasi:
data/processed/dataset_bersih.parquet

Hasil penyimpanan:
- Jumlah baris: 3.012.817
- Jumlah kolom: 5
- Ukuran file: 173,73 MB (sesuai tampilan notebook)

File Parquet telah berhasil dibaca kembali dan jumlah baris serta kolomnya telah diverifikasi.

### 7. Kesimpulan
Proses data cleaning berhasil mengatasi kolom dengan missing values 100%, menghapus duplikat berdasarkan kunci bisnis, dan melakukan evaluasi outlier tanpa menghilangkan teks yang masih berpotensi valid.

Dataset hasil cleaning disiapkan untuk digunakan pada Milestone 3, yaitu Exploratory Data Analysis (EDA) dan penyusunan insight.
